In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)


In [0]:
dbutils.widgets.text('catalog','commerce_stage_dev')
dbutils.widgets.text('schema','silver')
dbutils.widgets.text("env", "dev")

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
env = dbutils.widgets.get("env")

In [0]:
products_df = spark.table(f'commerce_raw_{env}.bronze.products')

In [0]:
display(products_df)

In [0]:
from pyspark.sql.functions import col
product_filter_df = products_df.filter(col("product_id").isNotNull())

In [0]:
product_clean_df = product_filter_df.distinct()

In [0]:
product_clean_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema}.products_temp1")

In [0]:
%sql
select * from products_temp1

In [0]:
spark.sql(f"""
CREATE TABLE if not EXISTS {catalog}.{schema}.products_stage_dedup as
select *  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY product_id
               ORDER BY ingestion_time DESC
           ) AS rn
    FROM products_temp1
)
WHERE rn = 1
""");


In [0]:
df = spark.sql(f'describe table extended commerce_stage_{env}.{schema}.products_stage_dedup')
display(df)

In [0]:
spark.sql(f"""
MERGE INTO commerce_stage_{env}.{schema}.product_stage ps
USING commerce_stage_{env}.{schema}.products_stage_dedup pd

ON ps.product_id = pd.product_id

WHEN MATCHED THEN
UPDATE SET
    ps.product_name = pd.product_name,
    ps.category = pd.category,
    ps.brand = pd.brand,
    ps.updated_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    product_id,
    product_name,
    category,
    brand,
    created_ts,
    updated_ts
)
VALUES (
    pd.product_id,
    pd.product_name,
    pd.category,
    pd.brand,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.products_temp1
          """)

In [0]:
spark.sql(f"""
          drop table if exists commerce_stage_{env}.{schema}.products_stage_dedup
          """)